# AquaCrop Calibration — Jablje & Rakičan

Calibrates four MaizeGDD parameters (`HI0`, `WP`, `CGC`, `CDC`) against observed grain yield (N3 fertilization, 1993–2010) using `scipy.optimize.differential_evolution`. Runs separately for Jablje (SiltLoam) and Rakičan (LoamySand).

Calibration typically takes 5–15 minutes per location depending on CPU speed.

In [ ]:
%load_ext autoreload
%autoreload 2

import json

import matplotlib.pyplot as plt
import pandas as pd

from aquacrop_slovenia import config
from aquacrop_slovenia.calibration import (
    MAIZE_GDD_DEFAULTS,
    calibrate,
    load_climate,
    load_co2,
    plot_calibration_results,
    run_single_season,
)

## 1. Smoke test — single season

Verify the data pipeline end-to-end using library defaults before running the optimizer.

In [ ]:
weather_jablje = load_climate("jablje")

yield_1993 = run_single_season(1993, MAIZE_GDD_DEFAULTS, "jablje", weather_jablje)
print(f"MaizeGDD defaults → 1993 Jablje yield: {yield_1993:.0f} kg/ha")
print(f"Expected range: 5,000 – 15,000 kg/ha")

## 2. CO2 data check

In [ ]:
for yr in [1993, 2000, 2010]:
    print(f"{yr}: {load_co2(yr):.2f} ppm")
# Expected: ~356, ~369, ~389 ppm

## 3. Observed yield (calibration target)

N3 fertilization (300 kg N/ha), grain only, averaged across management types A/B/C.

In [ ]:
def load_obs_yield(location, treatment="N3", period=(1993, 2010)):
    df = pd.read_csv(config.INTERIM_YIELD_DIR / f"maize-{location}.csv")
    series = (
        df[(df["fertilization"] == treatment) & (df["product"] == "Zrnje")]
        .groupby("year")["yield_kg_ha"]
        .mean()
    )
    return series[(series.index >= period[0]) & (series.index <= period[1])]

obs_jablje = load_obs_yield("jablje")
obs_rakican = load_obs_yield("rakican")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, obs_data, loc in zip(axes, [obs_jablje, obs_rakican], ["Jablje", "Rakičan"]):
    ax.bar(obs_data.index, obs_data.values, color="steelblue", edgecolor="white")
    ax.set_title(f"{loc} — N3 grain yield (calibration period)")
    ax.set_xlabel("Year")
    ax.set_ylabel("Yield (kg/ha)")
plt.tight_layout()
plt.show()

## 4. Calibrate Jablje

`differential_evolution` with `popsize=15`, up to 500 iterations. May take 5–15 minutes.

In [ ]:
result_jablje = calibrate("jablje")

print(f"Jablje RMSE : {result_jablje['rmse']:.0f} kg/ha")
print(f"Calibrated  : {result_jablje['params']}")
print(f"Converged   : {result_jablje['de_result'].success}")
print(f"Iterations  : {result_jablje['de_result'].nit}")
print(f"Evaluations : {result_jablje['de_result'].nfev}")
print(f"Valid seasons: {result_jablje['n_valid']} / {result_jablje['n_years']}")

## 5. Calibrate Rakičan

In [ ]:
result_rakican = calibrate("rakican")

print(f"Rakičan RMSE : {result_rakican['rmse']:.0f} kg/ha")
print(f"Calibrated   : {result_rakican['params']}")
print(f"Converged    : {result_rakican['de_result'].success}")
print(f"Iterations   : {result_rakican['de_result'].nit}")
print(f"Valid seasons: {result_rakican['n_valid']} / {result_rakican['n_years']}")

## 6. Calibration results

In [ ]:
fig = plot_calibration_results(result_jablje)
plt.show()

In [ ]:
fig = plot_calibration_results(result_rakican)
plt.show()

## 7. Summary

In [ ]:
summary = pd.DataFrame({
    "jablje": result_jablje["params"],
    "rakican": result_rakican["params"],
})
summary.loc["RMSE (kg/ha)"] = [result_jablje["rmse"], result_rakican["rmse"]]
summary.loc["n_years"] = [result_jablje["n_years"], result_rakican["n_years"]]
summary.loc["n_valid"] = [result_jablje["n_valid"], result_rakican["n_valid"]]
summary.round(5)

## 8. Save calibrated parameters

In [ ]:
out = {
    loc: {
        **res["params"],
        "planting_date": res["planting_date"],
        "treatment": res["treatment"],
        "rmse_kg_ha": res["rmse"],
        "n_years": res["n_years"],
        "n_valid": res["n_valid"],
    }
    for loc, res in [("jablje", result_jablje), ("rakican", result_rakican)]
}

out_path = config.DATA_DIR / "processed" / "calibration_params.json"
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(json.dumps(out, indent=2))
print(f"Saved → {out_path}")